In [1]:
import sys
import os
import matplotlib.pyplot as plt
import time

sys.path.append(os.path.abspath('../common'))
from DO_structs import *
from DO_graphics import *
from DO_server_graphics import *
from DO_searching import *

sys.path.append(os.path.abspath('../planners_library/'))
from sipp import *

%matplotlib inline

## 📊 Scalability Benchmark: Evaluating 6 Control Set Configurations

This section scales up the comparative analysis across six different control set primitives. We execute the Safe Interval Path Planning (SIPP) algorithm sequentially for each configuration, collect performance metrics, and render individual trajectory simulations.

Finally, we leverage `ffmpeg` to compile all six simulations into a single, perfectly synchronized **2x3 video grid** for unified performance visualization.


In [2]:
# Map & Environmnet
task_map = Map()
task_map.read_from_movingai_file("../maps/arena.map")

env_cs = ControlSet()
env_cs.load_primitives("../data/control_set.txt")

R = 1  # Ego-robot radius
environment = DynamicEnvironment(task_map, R)
environment.load_obstacles_from_dir("../dynamic_obstacles/arena_random/dynamic_obstacle_*.txt", env_cs, max_items=50)
environment.compile_safe_intervals()

In [3]:
# Configuration array defining identifiers and primitive source files
ALGORITHMS_CONFIG = [
    {'id': 'SIPP_Standard', 'file': '../data/control_set.txt'},
    {'id': 'SIPP_BigCS',    'file': '../data/extended_set.txt'},
    {'id': 'SIPP_2',        'file': '../data/primitives_2k_2.txt'},
    {'id': 'SIPP_3',        'file': '../data/primitives_2k_3.txt'},
    {'id': 'SIPP_4',        'file': '../data/primitives_2k_4.txt'},
    {'id': 'SIPP_5',        'file': '../data/primitives_2k_5.txt'},
]

# Shared spatial configurations for the current benchmark test case
start = DiscreteState(12, 40, 0)
goal = DiscreteState(43, 19, 0)

# Global rendering parameters shared across all video generators
common_kwargs = dict(
    task_map=task_map, 
    obstacle_set=environment.get_dynamic_obstacles_set(),
    max_time=570.0, dt=1.0, fps=35, show_path=True, obs_line_width=0.0,
    fast_draw=False, dpi=100, scale=1, workers=40
)

# Sequential execution loop across all configurations
for config in ALGORITHMS_CONFIG:
    algo_id = config['id']
    print(f"⏳ Processing configuration: {algo_id}...")
    
    # Dynamically load the control set primitives from file
    cs = ControlSet().load_primitives(config['file'])
    
    # Initialize and solve the SIPP search graph
    search_space = SIPPSearchSpace(start, 0.0, goal, cs, environment, R=0, A=0, position_only=True)
    success, path, steps, cost, ast = astar(search_space)
    
    if not success:
        print(f"⚠️ Warning: Path not found for {algo_id}")
        continue
    print(f"--> Path found! Cost: {cost}")
        
    # Reconstruct trajectory and compile the standalone video file
    robot_trajectory = DynamicObstacle(cs).set_parametres(R, start.i, start.j, start.theta, path)
    output_mp4 = f"../media/demo-grid_{algo_id}.mp4"
    
    render_simulation_parallel(output_file=output_mp4, robot=robot_trajectory, **common_kwargs)
    print(f"✅ Rendered standalone video: {output_mp4}")


⏳ Processing configuration: SIPP_Standard...
--> Path found! Cost: 552.8708569999999
🚀 Starting parallel rendering...
📊 Total frames: 571 | CPU Workers: 40


Rendering frames:   0%|          | 0/571 [00:00<?, ?it/s]

🎬 Assembling media file (Streaming Mode + HEVC)...


Saving MP4:   0%|          | 0/571 [00:00<?, ?it/s]

x265 [info]: HEVC encoder version 3.5+1-f0c1022b6
x265 [info]: build info [Linux][GCC 8.3.0][64 bit] 8bit+10bit+12bit
x265 [info]: using cpu capabilities: MMX2 SSE2Fast LZCNT SSSE3 SSE4.2 AVX FMA3 BMI2 AVX2
x265 [info]: Main profile, Level-3.1 (Main tier)
x265 [info]: Thread pool created using 64 threads
x265 [info]: Thread pool created using 64 threads
x265 [info]: Slices                              : 1
x265 [info]: frame threads / pool features       : 5 / wpp(14 rows)
x265 [info]: Coding QT: max CU size, min CU size : 64 / 8
x265 [info]: Residual QT: max TU size, max depth : 32 / 1 inter / 1 intra
x265 [info]: ME / range / subpel / merge         : hex / 57 / 2 / 3
x265 [info]: Keyframe min / max / scenecut / bias  : 25 / 250 / 40 / 5.00 
x265 [info]: Lookahead / bframes / badapt        : 20 / 4 / 2
x265 [info]: b-pyramid / weightp / weightb       : 1 / 1 / 0
x265 [info]: References / ref-limit  cu / depth  : 3 / off / on
x265 [info]: AQ: mode / str / qg-size / cu-tree  : 2 / 1.0 / 

...

Rendering frames:   0%|          | 0/571 [00:00<?, ?it/s]

🎬 Assembling media file (Streaming Mode + HEVC)...


Saving MP4:   0%|          | 0/571 [00:00<?, ?it/s]

x265 [info]: HEVC encoder version 3.5+1-f0c1022b6
x265 [info]: build info [Linux][GCC 8.3.0][64 bit] 8bit+10bit+12bit
x265 [info]: using cpu capabilities: MMX2 SSE2Fast LZCNT SSSE3 SSE4.2 AVX FMA3 BMI2 AVX2
x265 [info]: Main profile, Level-3.1 (Main tier)
x265 [info]: Thread pool created using 64 threads
x265 [info]: Thread pool created using 64 threads
x265 [info]: Slices                              : 1
x265 [info]: frame threads / pool features       : 5 / wpp(14 rows)
x265 [info]: Coding QT: max CU size, min CU size : 64 / 8
x265 [info]: Residual QT: max TU size, max depth : 32 / 1 inter / 1 intra
x265 [info]: ME / range / subpel / merge         : hex / 57 / 2 / 3
x265 [info]: Keyframe min / max / scenecut / bias  : 25 / 250 / 40 / 5.00 
x265 [info]: Lookahead / bframes / badapt        : 20 / 4 / 2
x265 [info]: b-pyramid / weightp / weightb       : 1 / 1 / 0
x265 [info]: References / ref-limit  cu / depth  : 3 / off / on
x265 [info]: AQ: mode / str / qg-size / cu-tree  : 2 / 1.0 / 

✅ Success! Render saved to: ../media/demo-grid_SIPP_5.mp4
🧹 Cleaning up temporary frames...
✅ Rendered standalone video: ../media/demo-grid_SIPP_5.mp4


In [4]:
import subprocess

print("🎬 Merging 6 video streams into a single 2x3 grid...")

# Define source video paths in grid order (Row 1: left, mid, right | Row 2: left, mid, right)
video_inputs = [
    "../media/demo-grid_SIPP_Standard.mp4",
    "../media/demo-grid_SIPP_BigCS.mp4",
    "../media/demo-grid_SIPP_2.mp4",
    "../media/demo-grid_SIPP_3.mp4",
    "../media/demo-grid_SIPP_4.mp4",
    "../media/demo-grid_SIPP_5.mp4"
]
output_grid_video = "../media/benchmark_summary_grid_2_3.mp4"

# Build FFMPEG command components line-by-line
ffmpeg_cmd = ["ffmpeg", "-y"]
for video in video_inputs:
    ffmpeg_cmd.extend(["-i", video])

# Configure the xstack filter layout coordinates for a 2x3 layout
# Note: input streams are mapped from 0 to 5. Positions are defined as horizontal_offset_vertical_offset
xstack_layout = "0_0|w0_0|w0+w1_0|0_h0|w3_h0|w3+w4_h0"

# Inject the scaling/rounding filter straight into the complex filtergraph chain
filter_complex_str = f"xstack=inputs=6:layout={xstack_layout}[grid]; [grid]scale=trunc(iw/2)*2:trunc(ih/2)*2[v]"

ffmpeg_cmd.extend([
    "-filter_complex", filter_complex_str,
    "-map", "[v]",
    "-c:v", "libx265",
    "-pix_fmt", "yuv420p",
    "-crf", "23",
    "-preset", "medium",
    output_grid_video
])

# Execute FFMPEG pipeline via a subprocess wrapper
result = subprocess.run(ffmpeg_cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

if result.returncode == 0:
    print(f"🎉 Success! Combined 2x3 grid video saved to: {output_grid_video}")
else:
    print("❌ FFMPEG execution failed. Error log:")
    print(result.stderr)


🎬 Merging 6 video streams into a single 2x3 grid...
🎉 Success! Combined 2x3 grid video saved to: ../media/benchmark_summary_grid_2_3.mp4


<div style="display: flex; gap: 20px; justify-content: center; align-items: center; width: 100%;">
    <div style="text-align: center; flex: 1;">
        <h4>Comparison</h4>
        <video id="video-extended" src="../media/benchmark_summary_grid_2_3.mp4" controls width="100%" muted></video>
    </div>
</div>